In [32]:
import numpy as np
import pandas as pd
from scipy.optimize import minimize, minimize_scalar

# ----------------------------------------------------
# Load points.csv
# ----------------------------------------------------
df = pd.read_csv("xy_data.csv")
points = df[["x","y"]].to_numpy()

# ----------------------------------------------------
# Parametric curve
# ----------------------------------------------------
def curve(t, angle, growth, offset):

    x = (
        t*np.cos(angle)
        - np.exp(growth*np.abs(t))*np.sin(0.3*t)*np.sin(angle)
        + offset
    )

    y = (
        42
        + t*np.sin(angle)
        + np.exp(growth*np.abs(t))*np.sin(0.3*t)*np.cos(angle)
    )

    return x,y

# ----------------------------------------------------
# Distance from ONE point to curve
# ----------------------------------------------------
def point_error(px,py,angle,growth,offset):

    def f(t):
        x,y = curve(t,angle,growth,offset)
        return (x-px)**2 + (y-py)**2

    res = minimize_scalar(
        f,
        bounds=(6,60),
        method="bounded"
    )

    return res.fun

# ----------------------------------------------------
# Objective
# ----------------------------------------------------
def objective(params):

    angle,growth,offset = params

    total = 0.0

    for px,py in points:
        total += point_error(px,py,angle,growth,offset)

    print(params,total)

    return total

# ----------------------------------------------------
# Initial guess (Equation 4)
# ----------------------------------------------------
initial = np.array([
    0.5235983050410543,
    0.02999999687304454,
    54.99999821278573
])

# ----------------------------------------------------
# Optimize
# ----------------------------------------------------
result = minimize(
    objective,
    initial,
    method="Nelder-Mead",
    options={
        "maxiter":500,
        "disp":True,
        "xatol":1e-15,
        "fatol":1e-15
    }
)
print("\nBest Parameters")
print("----------------")
print(f"angle  = {result.x[0]:.50f}")
print(f"growth = {result.x[1]:.50f}")
print(f"offset = {result.x[2]:.50f}")
print(f"Total squared error = {result.fun:.50f}")

[5.23598305e-01 2.99999969e-02 5.49999982e+01] 1.2046763333549763e-08
[5.49778220e-01 2.99999969e-02 5.49999982e+01] 941.4877412634478
[5.23598305e-01 3.14999967e-02 5.49999982e+01] 36.76632417639249
[5.23598305e-01 2.99999969e-02 5.77499981e+01] 3550.6104558700654
[5.41051582e-01 3.09999968e-02 5.22499983e+01] 5453.485095638061
[5.27961624e-01 3.02499968e-02 5.63749982e+01] 754.7731253312386
[5.00327269e-01 3.11666634e-02 5.59166648e+01] 1746.8946332617193
[5.37415483e-01 3.02916635e-02 5.52291649e+01] 181.6773325708014
[5.28446437e-01 3.09444412e-02 5.37777760e+01] 929.31416871646
[5.28082828e-01 3.04236079e-02 5.57256926e+01] 184.4886461058433
[5.28325234e-01 3.07708301e-02 5.44270816e+01] 268.2476673472106
[5.28143429e-01 3.05104135e-02 5.54010399e+01] 51.86439173917169
[5.12811210e-01 3.10486079e-02 5.50381927e+01] 202.70531437289603
[5.31264414e-01 3.04808996e-02 5.51814218e+01] 50.075410690030374
[5.24163920e-01 3.08101820e-02 5.47199056e+01] 51.002658286338324
[5.25158798e-01 3

In [ ]:
# Best Parameters
# ----------------
# angle  = 0.52359830444084520806313776120077818632125854492188
# growth = 0.02999999717091086925968568266398506239056587219238
# offset = 54.99999835238169509921135613694787025451660156250000
# Total squared error = 0.00000001203222843546113176927387744367947686185971

In [48]:
import numpy as np
import pandas as pd

# ============================================================
# Load dataset
# ============================================================

df = pd.read_csv("xy_data.csv")

x_coords = df["x"].to_numpy()
y_coords = df["y"].to_numpy()

# ============================================================
# Reconstruction Error (L1)
# ============================================================

def calculate_reconstruction_error(params, x_coords, y_coords):
    theta, m, x_shift = params

    # Translate coordinates
    tx = x_coords - x_shift
    ty = y_coords - 42.0

    # Rotate coordinates
    t = tx * np.cos(theta) + ty * np.sin(theta)
    v_obs = -tx * np.sin(theta) + ty * np.cos(theta)

    # Predicted curve
    v_pred = np.exp(m * np.abs(t)) * np.sin(0.3 * t)

    residual = np.abs(v_obs - v_pred)

    total_l1 = np.sum(residual)
    average_l1 = np.mean(residual)
    maximum_error = np.max(residual)

    return total_l1, average_l1, maximum_error


# ============================================================
# Parameters to Compare
# ============================================================

parameters = {

    "Old Parameters": (
        0.52359830504105429184918705359740209765275037604314,
   0.0299999968730445439046849998021571082063019275665,
  54.9999982127857265368220396339893341064453125
    ),

    "New Parameters": (
        0.52359830444084520806313776120077818632125854492188,
        0.02999999717091086925968568266398506239056587219238,
        54.99999835238169509921135613694787025451660156250000
    )

}

# ============================================================
# Evaluate
# ============================================================

results = []

for name, params in parameters.items():

    total_l1, average_l1, maximum_error = calculate_reconstruction_error(
        params,
        x_coords,
        y_coords
    )

    results.append({
        "Parameters": name,
        "Total L1": total_l1,
        "Average L1": average_l1,
        "Maximum Error": maximum_error
    })

results = pd.DataFrame(results)

results = results.sort_values("Total L1")

# ============================================================
# Display Results
# ============================================================

pd.set_option("display.precision", 18)

print("\n================== RESULTS ==================\n")
print(results.to_string(index=False))

best = results.iloc[0]

print("\n=============================================")
print(f"Best Parameters : {best['Parameters']}")
print(f"Total L1        : {best['Total L1']:.18f}")
print(f"Average L1      : {best['Average L1']:.18f}")
print(f"Maximum Error   : {best['Maximum Error']:.18f}")
print("=============================================")


================== RESULTS ==================

    Parameters                         Total L1                       Average L1                    Maximum Error
New Parameters 0.003838064148084554139811475082 0.000002558709432056369272945676 0.000017442724391081299017969286
Old Parameters 0.003840943087120290456992766082 0.000002560628724746860399529534 0.000017521407762011165232252097

Best Parameters : New Parameters
Total L1        : 0.003838064148084554
Average L1      : 0.000002558709432056
Maximum Error   : 0.000017442724391081
